<a href="https://colab.research.google.com/github/haneef5244/google-colab/blob/langgraph-reflection-agent/langgraph-reflection-agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# @title
from google.colab import userdata
import os

# Explicitly set environment variables to ensure they are picked up immediately
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = "langgraph-test"
os.environ["LANGCHAIN_TRACING"] = "true"
os.environ["TAVILY_API_KEY"]=userdata.get("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"]=userdata.get("GROQ_API_KEY")
os.environ["ANTHROPIC_API_KEY"]=userdata.get("ANTHROPIC_API_KEY")

In [3]:
import os

langchain_tracing_enabled = os.getenv("LANGCHAIN_TRACING")

if langchain_tracing_enabled == "true":
    print("Langsmith tracing is enabled.")
else:
    print("Langsmith tracing is not enabled or is set to a different value.")
print(f"Current value of LANGCHAIN_TRACING: {langchain_tracing_enabled}")

Langsmith tracing is enabled.
Current value of LANGCHAIN_TRACING: true


In [4]:
# @title
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 52 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (519 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directo

In [22]:
!nohup ollama serve > ollama.log 2>&1 &

In [6]:
!ollama pull qwen3.5:9b
!ollama pull huihui_ai/qwen3.5-abliterated:9b

In [1]:
!pip install langchain langchain-ollama python-dotenv langgraph langchain-groq langchain-tavily langchain-anthropic

In [7]:
# states.py
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage
from langgraph.graph import add_messages

# This class holds the schema of the state of the graph
class MessageGraph(TypedDict):
    # below annotation tells LangGraph that messages
    # is a list of BaseMessages (AIMessage, HumanMessages or SystemMessage)
    # and how to change state/update this data is to use add_messages method
    messages: Annotated[list[BaseMessage], add_messages]

In [9]:
from langchain_tavily import TavilySearch
from dotenv import load_dotenv

load_dotenv()

tools = [TavilySearch(max_results=5)]

In [2]:
# chains.py

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_ollama import ChatOllama
from langchain_groq import ChatGroq
from langchain_anthropic import ChatAnthropic

# from langchain_course.tools import tools

"""
    ChatPromptTemplate holds all the messages either from User or generated from the LLM
    e.g.
        chat_template = ChatPromptTemplate.from_messages(
            ("system", "You are a helpful AI bot. your name is {name})"),
            ("human", "Hello, how are you doing?"),
            ("ai", "I'm doing well, thanks!"),
            ("human", "{user_input}")
        )
        messages = chat_template.format_messages(name="Bob", user_input="What is your ....")

    MessagesPlaceHolder gives flexility to put placeholder for future messages that we're going to get
    e.g.
        human_prompt = "Summarize our conversation so far in {word_count} words."
        human_message_template = HummanMessagePromptTemplate.from_template(human_prompt)

        chat_prompt = ChatPromptTemplate.from_messages(
            [
                MessagesPlaceHolder(variable_name="conversation"),
                human_message_template
            ]
        )
"""

# act as critique - review the output and critize it (how it can be better)
reflection_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a data analyst expert reviewer. Provide critique to the user output so that it meets the expectations of the initial user prompt. "
            "Always provide a detailed recommendations. "
            "If the output looks fine, just return `1` as your output"
        ),
        # Below is the placeholder for messages (history)
        MessagesPlaceholder(variable_name="messages"),
    ]
)

# generation prompt
generation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a data analyst assistant."
            "You are provided with tools to get data for the user requests."
            "If the user provides critique, response with a revised version of your previous attempts."
        ),
        MessagesPlaceholder(variable_name="messages")
    ]
)

llm_base = ChatOllama(model="huihui_ai/qwen3.5-abliterated:9b", temperature=0.3)
llm_generation = llm_base.bind_tools(tools=tools).with_retry(stop_after_attempt=10, wait_exponential_jitter=True)
# llm_reflection = ChatOllama(model="huihui_ai/qwen3.5-abliterated:9b", temperature=0.3).with_retry(stop_after_attempt=10, wait_exponential_jitter=True)
llm_reflection = ChatAnthropic(model="claude-sonnet-4-5-20250929", temperature=0.3).with_retry(stop_after_attempt=10, wait_exponential_jitter=True)
# llm_reflection = llm_base.with_retry(stop_after_attempt=10, wait_exponential_jitter=True)
# llm_generation = ChatGroq(model="qwen/qwen3.8-27b").bind_tools(tools=tools).with_retry(stop_after_attempt=10, wait_exponential_jitter=True)
generation_chain = generation_prompt | llm_generation
reflection_chain = reflection_prompt | llm_reflection





NameError: name 'tools' is not defined

In [11]:
# @title
# nodes.py

from langchain_core.messages import HumanMessage
from langgraph.prebuilt import ToolNode

# from langchain_course.chains import generation_chain, reflection_chain
# from langchain_course.states import MessageGraph
# from langchain_course.tools import tools

# since we initialize our graph with MessageGraph,
# each nodes should receive state of schema MessageGraph
def generation_node(state: MessageGraph):
    print(" --> Inside Generation_Node")
    print(state)
    return {
        "messages": [generation_chain.invoke({ "messages": [*state["messages"]] })]
    }

# since we initialize our graph with MessageGraph,
# each nodes should receive state of schema MessageGraph
def reflection_node(state: MessageGraph):
    print(" --> Inside reflection_node")
    print(state)
    res = reflection_chain.invoke({ "messages": [*state["messages"]]})
    print(" --> Reflection node res = ")
    print(res)
    return { "messages": [HumanMessage(content=res.content)] }

tool_node = ToolNode(tools=tools)

In [23]:
from dotenv import load_dotenv
import os

from langgraph.graph import END, MessagesState, StateGraph
from langchain_core.messages import ToolMessage, AIMessage, HumanMessage

# google collab built-in debugger
import pdb

# from langchain_course.chains import generation_chain, reflection_chain
# from langchain_course.nodes import generation_node, reflection_node, tool_node
# from langchain_course.states import MessageGraph

load_dotenv()

GENERATE="generate"
REFLECT="reflect"
ACT="act"
LAST=-1

def should_continue(state: MessageGraph) -> str:
    last_human_message = next(
            (msg for msg in reversed(state["messages"]) if isinstance(msg, HumanMessage)),
            None
        )

    if last_human_message.content == "1":
        return END

    last_message = state["messages"][LAST]
    # if AI message contains tool_calls
    if last_message.tool_calls:
        return ACT

    # 3. If tool just finished -> GENERATE (synthesize search results)
    if isinstance(last_message, ToolMessage):
        pdb.set_trace()

        return GENERATE

    # 4. If AI produced text draft -> check if it's from REFLECT or GENERATE
    if isinstance(last_message, AIMessage):
        # Prevent infinite reflection loop
        if getattr(last_message, "name", None) == "reflect":
            return GENERATE  # Or END if approved
        return REFLECT

    return GENERATE



def main() -> None:
    print("Hello from langchain-course!")

    # since we initialize our graph with MessageGraph,
    # each nodes should receive state of schema MessageGraph
    builder = StateGraph(state_schema=MessageGraph)

    builder.add_node(GENERATE, generation_node)
    builder.add_node(REFLECT, reflection_node)
    builder.add_node(ACT, tool_node)
    # creates edge from the pre-existing START node to point to GENERATE
    # START -> GENERATE
    builder.set_entry_point(GENERATE)

    # Deterministic edge
    builder.add_edge(REFLECT, GENERATE)
    builder.add_edge(ACT, GENERATE)

    # After GENERATE node completes, should_continue function will always be called,
    # and the output of should_continue will decide whether to go to REFLECT node or the END node
    builder.add_conditional_edges(GENERATE, should_continue, path_map={
        REFLECT: REFLECT,
        ACT: ACT,
        END: END,
        GENERATE: GENERATE
    })

    # Graph gets built only after compiled
    graph = builder.compile()
    graph.get_graph().draw_mermaid_png(output_file_path="graph.png")

    res = graph.invoke({
        "messages": [
            {
                "role": "user",
                "content": """
                Generate me 10 contacts from any Malaysia Government agencies from the Learning and Development centre. Provide me with the sources of where you get this information as well.
                The list should contain the following:
                1) Name of the contact person
                2) Email or phone number of the contact person
                3) Department
                4) Agency name
                """
            }
        ]
    })

    print(res["messages"][-1].content)



if __name__ == "__main__":
    main()



Hello from langchain-course!
 --> Inside Generation_Node
{'messages': [HumanMessage(content='\n                Generate me 10 contacts from any Malaysia Government agencies from the Learning and Development centre. Provide me with the sources of where you get this information as well.\n                The list should contain the following:\n                1) Name of the contact person\n                2) Email or phone number of the contact person\n                3) Department\n                4) Agency name\n                ', additional_kwargs={}, response_metadata={}, id='e2515692-296f-4cdf-9f9b-305647bdc2ea')]}
 --> Inside Generation_Node
{'messages': [HumanMessage(content='\n                Generate me 10 contacts from any Malaysia Government agencies from the Learning and Development centre. Provide me with the sources of where you get this information as well.\n                The list should contain the following:\n                1) Name of the contact person\n              

KeyboardInterrupt: 